## Data Pipeline

- Dynamically pulls data from GitHub and API sources
    - *Note:* you may need to change the Wikipedia user session data for your project.
- Clean data
    - Check for null values and decide to impute or drop.
    - Check for class distributions.
    - Check how many languages are missing key features from the table in the cell below.
- Stack dataframes
    - Ensure languages aren't duplicated.
    - Save as CSV file.

### Sources

*The following citations represent the webpages or datasets from where the final dataset features were sourced.*

#### Endangered Languages Project (ELP/ELCAT)

Previous versions of this project utilized a CSV file from the Endangered Languages Project (ELP). Since then, ELP has shifted to hosting versions of their data on GitHub through the  ELCAT, a repository where they keep the dataset and each of its independent features as individual CSV files. This new data pipeline will dynamically pull the github repository to build a structured pandas dataframe, rather than working with the CSV from the earlier version of this project.

Catalogue of Endangered Languages. 2023. University of Hawaii at Manoa. http://www.endangeredlanguages.com

[ELCAT Github Link](https://github.com/cldf-datasets/elcat/tree/main)

#### Glottolog

Harald Hammarström, Robert Forkel, Martin Haspelmath, & Sebastian Bank. (2026). glottolog/glottolog: Glottolog database 5.3 (v5.3) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.18840935

[Glottolog GitHub Link](https://github.com/glottolog/glottolog?tab=readme-ov-file)

#### Wikipedia

Wikipedia contributors. (2026, April 12). Languages used on the Internet. In Wikipedia, The Free Encyclopedia. Retrieved 18:29, April 28, 2026, from https://en.wikipedia.org/w/index.php?title=Languages_used_on_the_Internet&oldid=1348337473

Wikipedia contributors. (2026, April 24). List of languages by total number of speakers. In Wikipedia, The Free Encyclopedia. Retrieved 18:30, April 28, 2026, from https://en.wikipedia.org/w/index.php?title=List_of_languages_by_total_number_of_speakers&oldid=1350848572

Wikipedia contributors. (2026, April 14). List of official languages by country and territory. In Wikipedia, The Free Encyclopedia. Retrieved 18:22, April 28, 2026, from https://en.wikipedia.org/w/index.php?title=List_of_official_languages_by_country_and_territory&oldid=1348824802

#### World Bank

World Bank, World Development Indicators. (2024). GDP per capita (current US$) [Data file]. Retrieved from https://data.worldbank.org/indicator/NY.GDP.PCAP.CD

World Bank, World Development Indicators. (2024). Individuals using the internet (% of population) [Data file]. Retrieved from https://data.worldbank.org/indicator/IT.NET.USER.ZS

World Bank, World Development Indicators. (2024). Urban population (% of total population) [Data file]. Retrieved from https://data.worldbank.org/indicator/SP.URB.TOTL.IN.ZS

World Bank, World Development Indicators. (2024). Urban population growth (annual %) [Data file]. Retrieved from https://data.worldbank.org/indicator/SP.URB.GROW

|Target feature|Source|Notes|
|---|---|---|
|endangerment_level|Glottolog + ELCAT|Contains rankings of both endangered and non-endangered languages from multiple sources (including UNESCO and ethnologue). Use as the source of truth. Will need to be compared to ELCAT to reconcile discrepencies.|

|Feature Name|Source|Notes|
|---|---|---|
|official_name|Glottolog + ELCAT + Wikipedia|Official name of the language. Will need to be compared across datasets to ensure consistency when joining.|
|iso_code|ELCAT + Glottolog|Currently labeled as "abbrv". Needs to be compared against glottolog iso codes when stacking the df. Wikipedia does not have iso codes, so will need to compare it against the name using regex standardization, possibly "other names" column from elp dataset. Use for stacking then drop before predictions.|
|speaker_number|ELCAT| Ordinal value, see https://github.com/cldf-datasets/elcat/blob/main/cldf/codes.csv for codes. This feature will be taken out to see if we can achieve high accuracy without it using the other features. This is beacuase number of speakers is a symptom, not a root cause.|
|speaker_number_trend|ELCAT|Ordinal value, see https://github.com/cldf-datasets/elcat/blob/main/cldf/codes.csv for codes.|
|transmission|ELCAT|Ordinal value, see https://github.com/cldf-datasets/elcat/blob/main/cldf/codes.csv for codes.|
domains_of_use|ELCAT|Ordinal value, see https://github.com/cldf-datasets/elcat/blob/main/cldf/codes.csv for codes.|
|lang_family|Glottolog|which language family a given language belongs to|
|lang_family_count|Glottolog|count of other languages in the same family|
|country|ELCAT + Wikipedia|list of countries where the language is spoken. Used for data cleaning and comparison, dropped before modeling|
|country_count|ELCAT + wikipedia|number of countries where the language is spoken|
|macroarea|ELCAT + Glottolog|replaces continent since it is more indicative of the language than a continent is|
|official|Wikipedia|binary 1 if the language is recognized as an official language in any country|
|regional|Wikipedia| binary 1 if the language is recognized as a regional language in any country|
|national|Wikipedia|binary 1 if the language is recognized as a national language in any country|
|minority|Wikipedia|binary 1 if the language is recognized as a minority language in any country|
|widely_spoken|Wikipedia|binary 1 if the language is recognized as widely spoken in any country|
|internet_usage|Wikipedia|binary 1 if the language is used on the list of used on the internet (shows everything used above 0.1%)|
|max_gdp_percapita|world bank| maximum gdp across the countries where the language is spoken|
|max_internet|World Bank|maximum internet usage across the countries where the language is spoken|
|max_urban|World Bank|maximum urban population % across the countries where the language is spoken|
|max_urban_growth|World Bank|maximum urban growth rate across countries where the language is spoken|
|avg_gdp_percapita|World Bank| average gdp per capita across the countries where the language is spoken|
|avg_urban|World Bank|average urban population % across the countries where the language is spoken|
|avg_internet|World Bank|average internet usage rate across the countries where the language is spoken|
|avg_urban_growth|World Bank|average urban growth rate across the countries where the language is spoken|


## Install Dependencies

In [ ]:
# Install packages if not already installed

#!pip install wbgapi
#!pip install pyglottolog
#!pip install wikipedia-api

In [1]:
# System
import os
import requests 

# Data Cleaning and Plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from rapidfuzz import process, fuzz

# Data APIs
import wikipediaapi as wiki
import wbgapi as wb
from pyglottolog import Glottolog


## Data Loading

*Dynamically pull data fom GitHub and API sources.*

### Endangered Languages Project (ELP/ELCAT)

*Note*: The updated GitHub repository for the Endangered Languages Catalogue has a slightly different format than the CSV that was previously used in this project. The repository now includes value estimates for the following:
- LEI (Language Endangerment Index): Represents the estimated endangerment level for each language.
- Speaker number: Ordinal variable represented by an integer for the estimated number of speakers.
- Speaker number trends: Ordinal variable represented by an integer for the estimated trends among communities (all members speak it, some members, etc.)
- Transmission: Ordinal variable represented by an integer for the estimated transmission from adults to children.
- Domains of use: Ordinal variable represented by an integer for which types of domains the language is used in (Government, Education, etc.)

In [2]:
# Pin latest commit

ELCAT_COMMIT = "main"

ELCAT_BASE_URL = f"https://raw.githubusercontent.com/cldf-datasets/elcat/{ELCAT_COMMIT}/cldf"

In [3]:
# Pull values.csv from ELCAT
PARAMS_TO_KEEP = ['speaker_number', 'transmission', 'speaker_number_trends', 'LEI', 'domains_of_use']

values = pd.read_csv(f"{ELCAT_BASE_URL}/values.csv")

# Filter to preferred values
preferred = values[(values['preferred'] == 'yes') & values['Parameter_ID'].isin(PARAMS_TO_KEEP)].copy()

print(preferred.groupby('Parameter_ID').size())

print('-'*50)

# Check for multiple preferred values per language
dupes = preferred.groupby(['Language_ID', 'Parameter_ID']).size()
print(f" Duplicate values found: {dupes[dupes > 1]}")

Parameter_ID
LEI                      3402
domains_of_use            584
speaker_number           3076
speaker_number_trends     778
transmission              942
dtype: int64
--------------------------------------------------
 Duplicate values found: Series([], dtype: int64)


In [4]:
# Create values dataframe
values_wide = preferred.pivot(index= 'Language_ID', columns= 'Parameter_ID', values= 'Value').reset_index()
values_wide.head()

Parameter_ID,Language_ID,LEI,domains_of_use,speaker_number,speaker_number_trends,transmission
0,21,"Endangered (100 percent certain, based on the ...",3,2,3,2
1,61,"Vulnerable (60 percent certain, based on the e...",NaN,1,NaN,0
2,101,"Endangered (100 percent certain, based on the ...",4,2,2,3
3,121,"Critically Endangered (60 percent certain, bas...",NaN,5,NaN,5
4,125,"Severely Endangered (100 percent certain, base...",4,3,5,4


In [5]:
# Load languages data
languages = pd.read_csv(f"{ELCAT_BASE_URL}/languages.csv")
languages.head()

,ID,Name,Macroarea,Latitude,Longitude,Glottocode,ISO639P3code,Comment,Countries,ELCatMacroareas,classification,endangerment,code_authorities,codes,alt_names
0,10009,Wampanoag,NaN,41.702591,-73.319119,wamp1249,wam,Massachusett-Narragansett was a complex of Sou...,US,North America,Algic,awakening,ISO 639-3,wam,Wôpanâak; Massachusett-Narragansett; Massach...
1,1001,Xipaya,NaN,-4.390200,-53.964800,xipa1240,xiy,NaN,BR,South America,Tupian,dormant,ISO 639-3,xiy,Shipaja; Xipaia; Shipaya; Chipaia; Xipáya; Šip...
2,1002,Yuchi,NaN,36.000900,-96.098800,yuch1247,yuc,NaN,US,North America,Isolate,critically endangered,ISO 639-3,yuc,Euchee
3,10029,Suabo,NaN,-2.100000,132.197000,suab1238,szp,NaN,ID,Southeast Asia,Trans-New Guinea,critically endangered,ISO 639-3,szp,Suabau; Inanwatan; Mirabo; Iagu; Solowat; Itig...
4,1003,Wichita,NaN,35.064700,-98.278800,wich1260,wic,NaN,US,North America,Caddoan,dormant,ISO 639-3,wic,Witchita


In [6]:
# Merge values and languages
elcat_df = pd.merge(values_wide, languages, 
                    left_on='Language_ID', 
                    right_on='ID', 
                    how='left').drop(columns= 'Language_ID')

elcat_df.head()

,LEI,domains_of_use,speaker_number,speaker_number_trends,transmission,ID,Name,Macroarea,Latitude,Longitude,Glottocode,ISO639P3code,Comment,Countries,ELCatMacroareas,classification,endangerment,code_authorities,codes,alt_names
0,"Endangered (100 percent certain, based on the ...",3,2,3,2,21,Namuyi,NaN,28.072700,101.956500,namu1246,nmy,NaN,CN,East Asia,Sino-Tibetan,endangered,ISO 639-3,nmy,Namuzi; 納木依; 納木義; 納木茲
1,"Vulnerable (60 percent certain, based on the e...",NaN,1,NaN,0,61,Himba,NaN,-18.056570,13.840657,zemb1238,dhm,NaN,NA AO,Africa,Niger-Congo,vulnerable,ISO 639-3,dhm,Dhimba; Dimba; Otjidhimba; Zemba; Tjimba; Simb...
2,"Endangered (100 percent certain, based on the ...",4,2,2,3,101,Kulaale,NaN,10.176552,18.566710,fani1244,fni,NaN,TD,Africa,Niger-Congo,endangered,ISO 639-3; Glottolog,fni; fani1244,Fagnia; Fanya; Fanyan; Fana; Fania; Fanian; Ma...
3,"Critically Endangered (60 percent certain, bas...",NaN,5,NaN,5,121,Potawatomi,NaN,39.325100,-95.849500,pota1247,pot,NaN,US CA,North America,Algic,critically endangered,ISO 639-3,pot,Pottawotomi; Bodéwadmi; Bodewadmi
4,"Severely Endangered (100 percent certain, base...",4,3,5,4,125,Hawaiian,NaN,21.894103,-160.161856,hawa1245,haw,NaN,US,North America,Austronesian,severely endangered,ISO 639-3,haw,'Ōlelo Hawai'i; 'Ōlelo Hawai'i Makuahine


### Glottolog Data

In [7]:
# Load data from github repository
GLOTTOLOG_PATH = "../Data/glottolog-5.3"

if not os.path.exists(GLOTTOLOG_PATH):
    !git clone --depth 1 --branch v5.3 https://github.com/glottolog/glottolog.git {GLOTTOLOG_PATH}

glottolog = Glottolog(GLOTTOLOG_PATH)

In [8]:
# Check data loaded correctly 
print(glottolog)

langs = list(glottolog.languoids())
print(f"Number of langoids loaded: {len(langs)}")

for lang in langs[:5]:
    print(f"ID: {lang.id}, Name: {lang.name}, Level: {lang.level}")

<Glottolog repos v5.3 at /Users/jordan/github/Machine-Learning-Endangered-Languages/Data-Cleaning/../Data/glottolog-5.3>
Number of langoids loaded: 27177
ID: musk1252, Name: Muskogean, Level: LanguoidLevel(ordinal=1, id='family', description='sub-grouping of languoids above the language level')
ID: maba1274, Name: Maban, Level: LanguoidLevel(ordinal=1, id='family', description='sub-grouping of languoids above the language level')
ID: guri1248, Name: Muno, Level: LanguoidLevel(ordinal=2, id='language', description='defined by mutual non-intellegibility')
ID: pawa1255, Name: Pawaia, Level: LanguoidLevel(ordinal=2, id='language', description='defined by mutual non-intellegibility')
ID: bora1262, Name: Boran, Level: LanguoidLevel(ordinal=1, id='family', description='sub-grouping of languoids above the language level')


### Wikipedia Data

*Note*: for these pages, we know that the first table is the one relevant to this project, but make sure to check the actual url if an error occurs or if the table does not look right.

In [9]:
# Initialize Wikipedia API session
session = requests.Session()
session.headers.update({
    "User-Agent": "Predicting Endangered Languages (jordanandersen@berkeley.edu)"
})

wiki_wiki = wiki.Wikipedia(user_agent='MyProjectName (merlin@example.com)', language='en')


In [10]:
# Check that required pages exist
official_languages = wiki_wiki.page('List of official languages by country and territory')
print(f"Official languages page exists: {official_languages.exists()}")

top_20_languages = wiki_wiki.page('List of languages by number of native speakers')
print(f"Top 20 languages page exists: {top_20_languages.exists()}")

internet_languages = wiki_wiki.page('List of languages used on the Internet')
print(f"Internet languages page exists: {internet_languages.exists()}")

Official languages page exists: True
Top 20 languages page exists: True
Internet languages page exists: True


In [11]:
def extract_language_data(page):
    """
    Extracts data tables from a Wikipedia page and returns a DataFrame.
    """
    url = page.fullurl
    print(f"Extracting data from page: {page.title}: {page.fullurl}")

    response = session.get(url)
    response.raise_for_status()

    # Identify all tables on page
    tables = pd.read_html(response.text)
    print(f"Number of tables found: {len(tables)}")

    # Extract first table (may need to adjust if page structure changes)
    if tables:
        return tables[0]
    else:
        print(f"No tables found on page {page.title}")
        return None

In [12]:
official_languages = extract_language_data(official_languages)
official_languages.head()

Extracting data from page: List of official languages by country and territory: https://en.wikipedia.org/wiki/List_of_official_languages_by_country_and_territory
Number of tables found: 9


,Country/Region,Number of official (including de facto),Official language(s),Regional language(s),Minority language(s),National language(s),Widely spoken
0,Abkhazia[a],2,Abkhaz Russian,NaN,Georgian,Abkhaz,NaN
1,Afghanistan[1][2][3],2,Persian (Dari) Pashto,Uzbek[b] Turkmen[b] Pashayi[b] Nuristani[b] Ba...,NaN,Persian (Dari) Pashto,Persian (Dari)
2,Albania[4],1,Albanian,NaN,Greek Macedonian Aromanian,NaN,Italian
3,Algeria[5],2,Arabic Berber,NaN,NaN,Arabic Berber,French
4,Andorra,1,Catalan[6],NaN,Spanish French Portuguese,NaN,NaN


In [13]:
top_20_languages = extract_language_data(top_20_languages)
top_20_languages.head()

Extracting data from page: List of languages by number of native speakers: https://en.wikipedia.org/wiki/List_of_languages_by_number_of_native_speakers
Number of tables found: 8


,Language,Native speakers (millions),Language family,Branch
0,Mandarin Chinese,988,Sino-Tibetan,Sinitic
1,Spanish,487,Indo-European,Romance
2,English,372,Indo-European,Germanic
3,Hindi,347,Indo-European,Indo-Aryan
4,Portuguese,252,Indo-European,Romance


In [14]:
internet_languages = extract_language_data(internet_languages)
internet_languages.head()

Extracting data from page: Languages used on the Internet: https://en.wikipedia.org/wiki/Languages_used_on_the_Internet
Number of tables found: 9


,Rank,Language,15 May 2023,3 December 2025
0,1,English,55.5%,49.3%
1,2,Spanish,5.0%,6.0%
2,3,German,4.3%,5.9%
3,4,Japanese,3.7%,5.1%
4,5,French,4.4%,4.5%


### World Bank Data

In [15]:
# Search for relevant indicators
wb.series.info(id=["SP.URB.GROW","SP.URB.TOTL.IN.ZS","IT.NET.USER.ZS","NY.GDP.PCAP.CD"])

id,value
NY.GDP.PCAP.CD,GDP per capita (current US$)
IT.NET.USER.ZS,Individuals using the Internet (% of population)
SP.URB.TOTL.IN.ZS,Urban population (% of total population)
SP.URB.GROW,Urban population growth (annual %)
,4 elements


In [16]:
# Load GDP per capita last 10 years (current US$)
gdp = wb.data.DataFrame("NY.GDP.PCAP.CD", time=range(2014, 2025), labels=True).reset_index()
gdp.head()

,economy,Country,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
0,ZWE,Zimbabwe,1372.915262,1387.126326,1408.139453,3445.449410,2270.895319,2184.329239,2059.674454,2613.605421,2536.400502,2195.224921,2497.203322
1,ZMB,Zambia,1707.485731,1295.877887,1239.085279,1483.465773,1463.899979,1258.986198,951.644317,1127.160779,1447.123101,1330.727806,1187.109434
2,YEM,"Yemen, Rep.",1430.164210,1362.173812,975.359417,811.165970,633.887202,NaN,NaN,NaN,NaN,NaN,NaN
3,PSE,West Bank and Gaza,3352.112595,3272.154324,3527.613824,3620.360487,3562.330943,3656.858271,3233.568638,3678.635657,3799.955270,3455.028529,2592.305912
4,VIR,Virgin Islands (U.S.),33045.364380,34007.352941,35324.974887,35365.069304,36663.208755,38633.529892,39787.374165,42571.077737,44320.909186,NaN,NaN


In [17]:
# Load % using internet last 10 years
internet = wb.data.DataFrame("IT.NET.USER.ZS", time=range(2014, 2025), labels=True).reset_index()
internet.head()

,economy,Country,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
0,ZWE,Zimbabwe,16.364740,22.742800,23.120001,24.400000,25.000000,26.588301,29.298565,32.5924,36.256500,38.674801,41.642799
1,ZMB,Zambia,6.500000,8.800000,10.300000,12.200000,14.299997,14.471900,14.645800,14.8219,15.000000,16.450701,17.101000
2,YEM,"Yemen, Rep.",22.549999,24.085400,24.579201,NaN,NaN,17.490999,NaN,NaN,NaN,NaN,NaN
3,PSE,West Bank and Gaza,53.665200,56.700001,59.900002,63.299999,64.399994,70.622584,76.181602,NaN,88.646906,86.637653,NaN
4,VIR,Virgin Islands (U.S.),50.070000,54.839137,59.608316,64.377494,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Load urban % population last 10 years
urban_pop = wb.data.DataFrame("SP.URB.TOTL.IN.ZS", time=range(2014, 2025), labels=True).reset_index()
urban_pop.head()

,economy,Country,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
0,ZWE,Zimbabwe,33.162808,33.561264,34.079946,34.701495,35.408554,36.183764,37.009767,37.869206,38.705521,39.291937,39.893372
1,ZMB,Zambia,41.563474,41.955510,42.344248,42.729598,43.111474,43.489786,43.864449,44.235373,44.598851,45.028585,45.470384
2,YEM,"Yemen, Rep.",31.862616,32.388318,32.951201,33.558691,34.210605,34.906756,35.702560,36.070495,36.322631,36.570798,36.807752
3,PSE,West Bank and Gaza,85.043334,85.218217,85.337739,85.378226,85.535113,85.813617,86.074118,86.334956,86.596142,86.857687,87.119601
4,VIR,Virgin Islands (U.S.),94.879707,94.941332,94.994109,95.036679,95.067678,95.085748,95.097402,95.160431,95.217151,95.274079,95.331220


In [19]:
# Load urban growth last 10 years (annual %)
urban_growth = wb.data.DataFrame("SP.URB.GROW", time=range(2014, 2025), labels=True).reset_index()
urban_growth.head()

,economy,Country,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
0,ZWE,Zimbabwe,2.161444,2.534322,2.921846,3.250231,3.504475,3.729247,3.916481,4.021655,3.890609,3.180794,3.299572
1,ZMB,Zambia,4.127701,4.058949,4.016361,3.973471,3.895745,3.835322,3.761976,3.657400,3.581982,3.753005,3.788191
2,YEM,"Yemen, Rep.",4.394924,4.676698,4.725123,4.839049,4.884359,4.980809,5.127399,3.769546,3.569918,3.690719,3.627941
3,PSE,West Bank and Gaza,2.609577,2.495896,2.386264,2.036110,2.716618,2.836860,2.789635,2.759625,2.727602,2.694855,2.661359
4,VIR,Virgin Islands (U.S.),-0.074222,-0.092855,-0.126225,-0.174433,-0.228791,-0.291410,-0.343694,-0.329989,-0.372918,-0.411322,-0.456226


## Data Cleaning & Feature Creation